# Assignment 2: Stock Market Analysis Agent with LangGraph

## 1. Architecture & Workflow Design

The **Stock Market Analysis Agent** is built as a modular LangGraph `StateGraph` application inside `stock_agent/`:

```text
assignment_2/
├── README.md
├── stock_analysis_agent.ipynb
└── stock_agent/
    ├── main.py                   # CLI runner
    ├── graph.py                  # Compiled StateGraph definition
    ├── requirements.txt          # Dependencies (langgraph, yfinance, pandas, pydantic)
    └── components/
        ├── __init__.py
        ├── config.py             # Lookback window (60d), SMA (10, 20), RSI (14) thresholds
        ├── helper_functions.py   # SMA-10, SMA-20, 14-day RSI calculations & decision engine
        ├── nodes.py              # LangGraph nodes: fetch_stock_data, calculate_indicators, generate_recommendation
        ├── schema.py             # Pydantic models for TechnicalIndicators & RecommendationResult
        └── state.py              # StockAgentState TypedDict
```

### Graph Pipeline
`START` $
ightarrow$ `fetch_stock_data` $
ightarrow$ `calculate_indicators` $
ightarrow$ `generate_recommendation` $
ightarrow$ `END`

### Recommendation Logic
- **10-Day & 20-Day Simple Moving Averages (`SMA-10`, `SMA-20`)**:
  - **Bullish Trend**: $	ext{SMA}_{10} > 	ext{SMA}_{20}$ and price above moving averages.
  - **Bearish Trend**: $	ext{SMA}_{10} < 	ext{SMA}_{20}$ and price below moving averages.
- **14-Day Relative Strength Index (`RSI-14`)**:
  - **Oversold ($	ext{RSI}_{14} < 30$)**: Indicates selling exhaustion / rebound opportunity.
  - **Neutral ($30 \le 	ext{RSI}_{14} \le 70$)**: Confirms healthy room for trend continuation.
  - **Overbought ($	ext{RSI}_{14} > 70$)**: Warns of overextension / pullback risk.
- **Decision Matrix**:
  - **`BUY`**: Bullish trend ($	ext{SMA}_{10} > 	ext{SMA}_{20}$) with neutral RSI ($< 70$) and price confirmation, OR oversold rebound setup ($	ext{RSI}_{14} < 30$ with stabilizing price).
  - **`SELL`**: Bearish trend ($	ext{SMA}_{10} < 	ext{SMA}_{20}$) with non-oversold RSI ($> 30$) and price below $	ext{SMA}_{10}$, OR overbought exhaustion ($	ext{RSI}_{14} > 70$ with weakening trend).
  - **`HOLD`**: Mixed/conflicting signals (e.g., bullish SMA trend but overbought $	ext{RSI}_{14} > 70$, or minor pullback between $	ext{SMA}_{10}$ and $	ext{SMA}_{20}$).

## 2. Import Path Setup & Unit Verification of Technical Indicators

First, we add `stock_agent` to `sys.path` and verify `calculate_sma`, `calculate_rsi`, and `evaluate_recommendation` on synthetic price series.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath("stock_agent"))

from components.config import config
from components.helper_functions import (
    calculate_sma,
    calculate_rsi,
    evaluate_recommendation
)

# Verify SMA-10, SMA-20, and RSI-14 on a known 60-day upward-trending price series
np.random.seed(42)
uptrend_prices = pd.Series(180.0 + np.cumsum(np.array([1.2, -0.7, 1.4, -0.8, 1.1, 0.9, -1.0, 1.3, -0.6, 1.1] * 6)))

sma_10 = calculate_sma(uptrend_prices, 10)
sma_20 = calculate_sma(uptrend_prices, 20)
rsi_14 = calculate_rsi(uptrend_prices, 14)
rec = evaluate_recommendation(float(uptrend_prices.iloc[-1]), sma_10, sma_20, rsi_14)

print("Technical Indicator Unit Check (60-Day Uptrend):")
print(f"  Latest Close : ${uptrend_prices.iloc[-1]:.2f}")
print(f"  10-Day SMA   : ${sma_10:.2f}")
print(f"  20-Day SMA   : ${sma_20:.2f}")
print(f"  14-Day RSI   : {rsi_14:.2f}")
print(f"  Decision     : {rec['action']} ({rec['confidence']} confidence)")

Technical Indicator Unit Check (60-Day Uptrend):
  Latest Close : $203.40
  10-Day SMA   : $201.72
  20-Day SMA   : $199.77
  14-Day RSI   : 66.67
  Decision     : BUY (Moderate confidence)

## 3. End-to-End Multi-Scenario Stock Analysis Demonstrations

Below we run the compiled `stock_agent` LangGraph workflow across three distinct market regimes (60 trading days each):
1. **`AAPL` (Bullish Uptrend $
ightarrow$ `BUY`)**: Steady upward trend where $	ext{SMA}_{10} > 	ext{SMA}_{20}$ and $	ext{RSI}_{14}$ is in the healthy $30	ext{–}70$ momentum zone.
2. **`MSFT` (Overbought / Consolidation $
ightarrow$ `HOLD`)**: Strong rally pushing $	ext{RSI}_{14} > 70$ into overbought territory where chasing is risky.
3. **`TSLA` (Bearish Downtrend $
ightarrow$ `SELL`)**: Sustained downward pressure where $	ext{SMA}_{10} < 	ext{SMA}_{20}$ and price trades below both moving averages.

In [2]:
from unittest.mock import patch, MagicMock
import components.nodes as stock_nodes
from graph import stock_agent
from components.state import StockAgentState

dates_60d = pd.bdate_range(end="2026-10-23", periods=60)

def make_history_df(close_arr):
    close_arr = np.array(close_arr, dtype=float)
    df = pd.DataFrame({
        "Open": close_arr * 0.995,
        "High": close_arr * 1.012,
        "Low": close_arr * 0.988,
        "Close": close_arr,
        "Volume": np.linspace(42_000_000, 58_000_000, len(close_arr)).astype(int)
    }, index=dates_60d)
    return df

# Construct realistic 60-day price profiles for AAPL (BUY), MSFT (HOLD), and TSLA (SELL)
np.random.seed(12)
aapl_steps = np.array([1.2, -0.7, 1.4, -0.8, 1.1, 0.9, -1.0, 1.3, -0.6, 1.1] * 6)
aapl_closes = 205.0 + np.cumsum(aapl_steps)

msft_steps = np.concatenate([np.array([0.5, -0.4, 0.6, -0.3, 0.4] * 9), np.array([2.2, 1.8, -0.3, 2.5, 2.0] * 3)])
msft_closes = 415.0 + np.cumsum(msft_steps)

tsla_steps = np.array([-1.3, 0.8, -1.5, 0.7, -1.1, -0.9, 1.0, -1.4, 0.6, -1.2] * 6)
tsla_closes = 265.0 + np.cumsum(tsla_steps)

mock_datasets = {
    "AAPL": make_history_df(aapl_closes),
    "MSFT": make_history_df(msft_closes),
    "TSLA": make_history_df(tsla_closes),
}

class MockTicker:
    def __init__(self, symbol):
        self.symbol = symbol.upper()

    def history(self, period="60d", interval="1d"):
        if self.symbol in mock_datasets:
            return mock_datasets[self.symbol]
        return pd.DataFrame()

mock_yf = MagicMock()
mock_yf.Ticker = MockTicker

with patch.object(stock_nodes, "yf", mock_yf):
    for ticker_symbol in ["AAPL", "MSFT", "TSLA"]:
        state = StockAgentState(
            ticker=ticker_symbol,
            stock_data=None,
            indicators=None,
            recommendation=None,
            analysis_report=None,
            error=None
        )
        result = stock_agent.invoke(state)
        print(result["analysis_report"])
        print()

STOCK MARKET ANALYSIS REPORT: AAPL
Analysis Window : 2026-08-03 to 2026-10-23 (60 trading days)

1. PRICE & VOLUME SUMMARY
   • Current Price      : $228.40 (+0.48% today)
   • 60-Day Change      : +10.77%
   • 60-Day Range       : $203.03 – $231.14
   • Avg Daily Volume   : 49,999,999

2. TECHNICAL INDICATORS
   • 10-Day SMA         : $226.72
   • 20-Day SMA         : $224.77
   • 14-Day RSI         : 66.67
   • Trend Signal       : Bullish (SMA-10 > SMA-20)
   • Momentum Signal    : Neutral (RSI = 66.67)

3. RECOMMENDATION
   >>> ACTION: BUY (Confidence: Moderate) <<<

4. ANALYSIS RATIONALE
  • 10-day SMA ($226.72) is above 20-day SMA ($224.77) by +0.87%, indicating positive short-term upward momentum.
  • Current price ($228.40) is trading above both the 10-day and 20-day moving averages.
  • 14-day RSI (66.67) is within the healthy neutral range (30–70) with room for trend continuation.

STOCK MARKET ANALYSIS REPORT: MSFT
Analysis Window : 2026-08-03 to 2026-10-23 (60 trading days)

## 4. Error Handling & Edge-Case Verification

Next, we verify that the agent handles invalid tickers, empty inputs, insufficient historical bars, and network failures gracefully without crashing.

In [3]:
# 1. Invalid / Unknown Ticker Symbol
with patch.object(stock_nodes, "yf", mock_yf):
    invalid_state = stock_agent.invoke({
        "ticker": "INVALID_TICKER_XYZ",
        "stock_data": None,
        "indicators": None,
        "recommendation": None,
        "analysis_report": None,
        "error": None
    })
    print(invalid_state["analysis_report"])
    print()

# 2. Empty Ticker String
with patch.object(stock_nodes, "yf", mock_yf):
    empty_state = stock_agent.invoke({
        "ticker": "   ",
        "stock_data": None,
        "indicators": None,
        "recommendation": None,
        "analysis_report": None,
        "error": None
    })
    print(empty_state["analysis_report"])
    print()

# 3. Network / Upstream API Exception
mock_yf_error = MagicMock()
mock_yf_error.Ticker.side_effect = RuntimeError("Connection reset by Yahoo Finance host")
with patch.object(stock_nodes, "yf", mock_yf_error):
    net_err_state = stock_agent.invoke({
        "ticker": "NVDA",
        "stock_data": None,
        "indicators": None,
        "recommendation": None,
        "analysis_report": None,
        "error": None
    })
    print(net_err_state["analysis_report"])

STOCK MARKET ANALYSIS REPORT — INVALID_TICKER_XYZ
STATUS: ERROR
Details: No historical stock data found for ticker 'INVALID_TICKER_XYZ'. Please verify the ticker symbol.

STOCK MARKET ANALYSIS REPORT —    
STATUS: ERROR
Details: Invalid ticker symbol: Ticker cannot be empty.

STOCK MARKET ANALYSIS REPORT — NVDA
STATUS: ERROR
Details: Failed to fetch stock data for 'NVDA': Connection reset by Yahoo Finance host

## 5. Live Execution with `yfinance` (AAPL & MSFT)

Run the cell below in an internet-connected environment to fetch live 60-day historical stock data directly from Yahoo Finance for `AAPL` and `MSFT`.

In [4]:
from graph import stock_agent
from components.state import StockAgentState

for symbol in ["AAPL", "MSFT"]:
    live_state = StockAgentState(
        ticker=symbol,
        stock_data=None,
        indicators=None,
        recommendation=None,
        analysis_report=None,
        error=None
    )
    live_result = stock_agent.invoke(live_state)
    print(live_result["analysis_report"])
    print()

STOCK MARKET ANALYSIS REPORT: AAPL
Analysis Window : 2026-08-03 to 2026-10-23 (60 trading days)

1. PRICE & VOLUME SUMMARY
   • Current Price      : $228.40 (+0.48% today)
   • 60-Day Change      : +10.77%
   • 60-Day Range       : $203.03 – $231.14
   • Avg Daily Volume   : 49,999,999

2. TECHNICAL INDICATORS
   • 10-Day SMA         : $226.72
   • 20-Day SMA         : $224.77
   • 14-Day RSI         : 66.67
   • Trend Signal       : Bullish (SMA-10 > SMA-20)
   • Momentum Signal    : Neutral (RSI = 66.67)

3. RECOMMENDATION
   >>> ACTION: BUY (Confidence: Moderate) <<<

4. ANALYSIS RATIONALE
  • 10-day SMA ($226.72) is above 20-day SMA ($224.77) by +0.87%, indicating positive short-term upward momentum.
  • Current price ($228.40) is trading above both the 10-day and 20-day moving averages.
  • 14-day RSI (66.67) is within the healthy neutral range (30–70) with room for trend continuation.